# init

In [ ]:
from prompt import (
    ZeroShotPlannerPrompt, 
    OneShotPlannerPrompt, 
    QagZeroShotPlannerPrompt,
    QagOneShotPlannerPrompt,
    )
from typing import Any, Annotated, Literal, List, Optional, Dict
from pydantic import Field, BaseModel, ConfigDict
from openai import OpenAI
from pydantic import BaseModel, ConfigDict
from transformers import AutoTokenizer

# llm testing

In [ ]:
from openai import OpenAI
from pydantic import BaseModel, ConfigDict
from transformers import AutoTokenizer

class Answer(BaseModel):
    model_config = ConfigDict(extra="forbid")

    value: int
    explanation: str


client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="EMPTY",
)

model = "openai/gpt-oss-120b"
tokenizer = AutoTokenizer.from_pretrained(model)
chat_result = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "user",
            "content": "What is 2 + 2? Return the requested object. Reply with prose, do not use JSON, and add text before and after the answer.",
        }
    ],
    temperature=0,
    extra_body={
        "structured_outputs": {
            "json": Answer.model_json_schema(),
        }
    },
)

completion_result = client.completions.create(
    model=model,
    prompt=tokenizer.apply_chat_template(
        [
            {
                "role": "user",
                "content": "What is 2 + 2? Return the requested object.Reply with prose, do not use JSON, and add text before and after the answer.",
            }
        ],
        tokenize=False,
        add_generation_prompt=True,
    ),
    temperature=0,
    extra_body={
        "structured_outputs": {
            "json": Answer.model_json_schema(),
        }
    },
)

In [ ]:
print(chat_result.choices[0].message.content)

In [ ]:
print(completion_result.choices[0].text)

# ReWOO

In [ ]:
class EvidenceReference(BaseModel):
    model_config = ConfigDict(extra="forbid", frozen=True)
    ref: str = Field(
        description="Reference to an earlier evidence result, such as '#E1'",
        pattern=r"^#E[1-9]\d*$",
    )

    @property
    def index(self) -> int:
        return int(self.ref[2:])

ToolArgumentValue = (
    str
    | int
    | float
    | bool
    | list[str]
    | EvidenceReference
    | None
)

class PlannerStep(BaseModel):
    plan: str = Field(
        min_length=1,
        description="What this step does and why it is needed."
    )
    evidence_id: str = Field(
        description="Unique evidence identifier produced by this step",
        pattern=r"^#E[1-9]\d*$",
        examples=["#E1"]
    )
    tool_name: str = Field(
        min_length=1,
        description="Exact name of the tool to invoke.",
    )
    tool_arguments: dict[str, ToolArgumentValue] = Field(
        default_factory=dict,
        description=(
            "Arguments passed to the tool. Values may be literals, lists, "
            "null, or references to earlier evidence."
        ),
    )
    dependencies: Optional[list[EvidenceReference]] = Field(
        min_length=1,
        description="Earlier evidence results needed by this step"
    )

class PlannerResponse(BaseModel):
    objective: str = Field(
        min_length=1,
        description="The objective the plan is intended to solve"
    )
    steps: list[PlannerStep] = Field(
        min_length=1,
        description="Ordered tool execution plan"
    )

In [ ]:
tool_desc = """
This is list of the available tool and tool sets and their descriptions:

Case Handling Tools:
1. `get_cases_by_cohort_filter`: -> CaseSet 
A tool to query the GDC API for cases matching a given cohort filter.
Args:
    cohort_filter: GDC cohort filter
2. `compute_case_intersection`: -> CaseSet
A tool to compute the intersection between two case sets.
Args:
    case_set_A: CaseSet
    case_set_B: CaseSet
3. `compute_case_union`: -> CaseSet
A tool to compute the union between two case sets.
Args:
    case_set_A: CaseSet
    case_set_B: CaseSet

Cohort Creation Tools:
1. `construct_cohort_filter`: -> CohortFilter
A tool to construct a GDC cohort filter for case metadata.
The resulting filter is used to retrieve actual case counts using GDC API
Args:
    project: GDC project IDs. e.g ["TCGA-BRCA", "TCGA-LUAD"]
    primary_site: Primary site of a disease e.g. ["Lung", "Brain"]
    disease_type: Type of malignant disease e.g. ["adenomas and adenocarcinomas", "leukemia, nos"]
    sex: A person's sex at birth e.g. any / many of ["male", "female", "unknown"],
    min_age : Lower end of the age range, inclusive. Age at the time of diagnosis, expressed in years. e.g 34
    max_age : Higher end of the age range, inclusive. Age at the time of diagnosis, expressed in years. e.g 76
    smoking_status: Self reported current smoking status or smoking history of a patient. e.g. ["lifelong non-smoker", "current smoker"]

Case Availability Tools:
1. `get_cases_with_available_genomic_data`: -> CaseSet
A tool to retrieve cases with available data for a given genomic data type, for example 'ssm'.
This tool should always be used with a cohort filter, as data availability without a specific cohort can be slow to compute.
Some user queries may necessarily require computing data availability over all cases.
While such queries should be avoid if possible, if you absolutely must compute availability over all cases, create an empty cohort filter first.
Args:
    genomic_data_type: Valid options are "ssm", "cnv", "msi", "gene_expression"
    cohort_filter: GDC cohort Filter

Variant Tools:
1. `get_simple_somatic_mutation_occurrences`: -> CaseSet
A tool to query the GDC API for cases with simple somatic mutations (SSMs).
This tool searches within a specific cohort for SSMs within a given gene, optionally resulting in an amino acid (AA) change.
Args:
    cohort_filter: GDC cohort filter
    gene: Gene to query, for example, "BRAF"
    aa_change: Amino acid change to query (in HGVS protein notation), for example 'V600E'.

2. `get_copy_number_variant_occurrences`: -> CaseSet
A tool to query the GDC API for cases with copy number variations (CNVs).
This tool searches within a specific cohort for CNVs, optionally within a given gene, optionally for a specific change type.
If only the gene is specified, this tool retrieves all cases in the cohort with any change type for the given gene.
If only the change type is specified, this tool retrieves all cases in the cohort with the given change type in any gene.
It is an error to use this tool without at least one of gene or change type.
Args:
    cohort_filter: GDC Cohort Filter
    gene: Gene to query, for example, "BRAF"
    cnv_change: Copy number variation change to query. Valid options are 'gain', 'amplification', 'heterozygous deletion', or 'homozygous deletion'."

3. `get_microsatellite_instability_occurrences`: -> CaseSet
A tool to query the GDC API for cases with microsatellite instability (MSI).
This tool searches within a specific cohort for either MSI or MSS status.
Args:
    cohort_filter: GDC Cohort Filter
    msi_status: Microsatellite instability status. Valid options are 'msi' for microsatellite instable or 'mss' for microsatellite stable.

Gene Expression Tools
1. `get_gene_expression_statistics`:
A tool to query the GDC API for gene expression statistics for a list of genes within a set of cases.
Gene expression values are normalized either using FPKM-UQ or Median-centered log2(FPKM-UQ+1)
Args:
    case_set: Set of case IDs
    genes: A list of gene names to retrieve statistics for. Example ["BRAF", "BRCA"]
    normalization: Gene expression normalization method. Valid options are 'uqfpkm' and 'median_centered_log2_uqfpkm'

2. `get_most_varaibly_expressed_genes`:
A tool to query GDC API for the top-k most variably expressed genes within a set of cases.
Args:
    case_set: Set of case IDs
    top_k : An integer value representing the top k most varaibly expressed genes. Example '5'.
"""

task = "What is the prevalence of BRAF SSMs in lung cancer?"

zero_shot_prompt = ZeroShotPlannerPrompt.format(tool_description=tool_desc, task=task)
one_shot_prompt = OneShotPlannerPrompt.format(tool_description=tool_desc, task=task)
qag_zero_shot_prompt = QagZeroShotPlannerPrompt.format(tool_description=tool_desc, task=task)
qag_one_shot_prompt = QagOneShotPlannerPrompt.format(tool_description=tool_desc, task=task)

In [ ]:
tokens = tokenizer.encode(zero_shot_prompt)
print(len(tokens))

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="unused",  # Required by the client, unless you configured a real key
)

## q1

In [ ]:
q1_zero_shot_chat_result = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "user",
            "content": zero_shot_prompt,
        }
    ],
    temperature=1.0,
    max_tokens=2048,
    seed=42,
    extra_body={
        "structured_outputs": {
            "json": PlannerResponse.model_json_schema(),
        }
    },
)

In [ ]:
import json
json.loads(q1_zero_shot_chat_result.choices[0].message.content)

## q2

In [ ]:
task = "What is the prevalence of BRAF SSMs in lung cancer cases of non smoking women over 40?"
q2_zero_shot_prompt = QagZeroShotPlannerPrompt.format(tool_description=tool_desc, task=task)
q2_zero_shot_chat_result = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "user",
            "content": q2_zero_shot_prompt,
        }
    ],
    temperature=1.0,
    max_tokens=2048,
    extra_body={
        "structured_outputs": {
            "json": PlannerResponse.model_json_schema(),
        }
    },
)

In [ ]:
import json
json.loads(q2_zero_shot_chat_result.choices[0].message.content)

In [ ]:
q2_zero_shot_chat_result.choices[0].message.content

## q3

In [ ]:
task = "What is the average gene expression for TP53 and MUC16 in cases with lung adenocarcinoma?"
q3_zero_shot_prompt = QagZeroShotPlannerPrompt.format(tool_description=tool_desc, task=task)
q3_zero_shot_chat_result = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "user",
            "content": q3_zero_shot_prompt,
        }
    ],
    temperature=1.0,
    max_tokens=2048,
    extra_body={
        "structured_outputs": {
            "json": PlannerResponse.model_json_schema(),
        }
    },
)

In [ ]:
json.loads(q3_zero_shot_chat_result.choices[0].message.content)

## q4

In [ ]:
task = "What is the average gene expression of BRAF in TCGA-BRCA?"
q4_zero_shot_prompt = QagZeroShotPlannerPrompt.format(tool_description=tool_desc, task=task)
q4_zero_shot_chat_result = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "user",
            "content": q4_zero_shot_prompt,
        }
    ],
    temperature=1.0,
    max_tokens=2048,
    extra_body={
        "structured_outputs": {
            "json": PlannerResponse.model_json_schema(),
        }
    },
)


In [ ]:
json.loads(q4_zero_shot_chat_result.choices[0].message.content)

## q5

In [ ]:
task = "List the most variably expressed genes in APOLLO-LUAD."
q5_zero_shot_prompt = QagZeroShotPlannerPrompt.format(tool_description=tool_desc, task=task)
q5_zero_shot_chat_result = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "user",
            "content": q5_zero_shot_prompt,
        }
    ],
    temperature=1.0,
    max_tokens=2048,
    extra_body={
        "structured_outputs": {
            "json": PlannerResponse.model_json_schema(),
        }
    },
)

In [ ]:
json.loads(q5_zero_shot_chat_result.choices[0].message.content)

## q6

In [ ]:
task = "Compare the expression of TP53, MUC16, and BRAF in APOLLO-LUAD."
q6_zero_shot_prompt = QagZeroShotPlannerPrompt.format(tool_description=tool_desc, task=task)
q6_zero_shot_chat_result = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "user",
            "content": q6_zero_shot_prompt,
        }
    ],
    temperature=1.0,
    max_tokens=2048,
    extra_body={
        "structured_outputs": {
            "json": PlannerResponse.model_json_schema(),
        }
    },
)

In [ ]:
json.loads(q6_zero_shot_chat_result.choices[0].message.content)

In [ ]:
reasoning = [
    """
    We need to compute prevalence = number of cases with BRAF simple somatic mutations (SSM) divided by number of cases with any SSM (i.e., cases with available SSM data) in the cohort defined: lung cancer, non smoking women over 40. Need to define cohort filter: primary_site probably "Lung". Disease_type? Could be lung cancer general. We'll maybe leave disease_type null. Sex female. min_age 40 (inclusive). smoking_status "lifelong non-smoker". Perhaps also need to filter by project? Not specified, probably all projects.

    Steps:

    construct_cohort_filter with primary_site ["Lung"], sex ["female"], min_age 40, smoking_status ["lifelong non-smoker"].
    get_cases_by_cohort_filter to get case set for this cohort.
    get_simple_somatic_mutation_occurrences with cohort_filter ID from step 1? Actually the tool expects cohort_filter ID, not case set. So we need to use the cohort filter ID directly (the returned ID from construct_cohort_filter). So step1 returns a filter ID; we can use that in subsequent tools.
    get_simple_somatic_mutation_occurrences with gene BRAF, cohort_filter = filter ID.
    get_cases_with_available_genomic_data for ssm, cohort_filter = filter ID (to get denominator).
    get_case_set_size for both case_set_id (BRAF SSM) and case_set_id (SSM available) to get counts.
    Compute prevalence = BRAF_SSM_cases / total_SSM_cases.
    Let's do it.
    """,
    """
    We need to answer average gene expression for TP53 and MUC16 in cases with lung adenocarcinoma. Steps:
    Construct a cohort filter for lung adenocarcinoma. Need primary_site "Lung" and disease_type likely "adenocarcinoma". Use construct_cohort_filter with primary_site ["Lung"], disease_type ["adenocarcinoma"] maybe. The tool expects disease_type strings like "adenocarcinoma"? We'll assume. Or could be "Adenomas and Adenocarcinomas". Might be ambiguous. But we can try. Use construct_cohort_filter.
    Get cases by cohort filter.
    Get cases with available gene expression data in that cohort (genomic_data_type "gene_expression").
    Get gene expression statistics for genes TP53 and MUC16, normalization maybe "uqfpkm" or "median_centered_log2_uqfpkm". Use one, maybe default? Must specify. Use "uqfpkm".
    Then compute average? The tool returns statistics, likely includes median/mean? Not sure. It may return mean expression. We'll just output whatever it gives.
    """,
    """
    We need average gene expression of BRAF in TCGA-BRCA. Steps:
    Construct cohort filter for project TCGA-BRCA.
    Get cases with available gene expression data (genomic_data_type gene_expression) within that cohort.
    Then get gene expression statistics for gene BRAF, normalization maybe default? Must specify normalization: either uqfpkm or median_centered_log2_uqfpkm. User didn't specify, choose one, maybe default to uqfpkm? Could ask clarification but I think we can pick one and note.
    """,
    """
    User wants "most variably expressed genes in APOLLO-LUAD". Need to interpret "APOLLO-LUAD". Likely a GDC project ID? APOLLO-LUAD is a specific dataset maybe a project. We need to get cohort filter for project APOLLO-LUAD, then get cases, then get most variably expressed genes. Use construct_cohort_filter with project ["APOLLO-LUAD"], no other filters. Then get_cases_by_cohort_filter to retrieve case set. Then get_most_variably_expressed_genes with that case_set_id and default top_k? It wants list, maybe top few. Might want default (1) but ask "List the most variably expressed genes" maybe top 10? Could ask clarification but default top_k=1 gives one gene. Could set top_k e.g., 10. But the tool default is 1, but we can provide parameter top_k. Let's assume they want top few; we could pick top_k=10.    
    """,
    """
    User asks: "Compare the expression of TP53, MUC16, and BRAF in APOLLO-LUAD."

    We need expression data (gene expression) for those genes in the cohort APOLLO-LUAD. Likely a GDC project named APOLLO-LUAD.

    We need to get case set for cohort APOLLO-LUAD via construct_cohort_filter then get_cases_by_cohort_filter, then get_gene_expression_statistics for the genes TP53, MUC16, BRAF. Need normalization method. Probably use "uqfpkm" or "median_centered_log2_uqfpkm". Choose one, maybe median centered log2 for comparability. Provide stats: likely mean, median, etc. Not sure what fields tool returns. We'll just request and report stats.

    We need to first construct cohort filter with project = ["APOLLO-LUAD"]. Use construct_cohort_filter.

    Then get_cases_by_cohort_filter. Then get_gene_expression_statistics with genes list.

    Proceed.
    """
]

# Planner

In [ ]:
from __future__ import annotations
from dataclasses import dataclass, field
from collections import defaultdict

@dataclass
class PlanDag:
    steps: dict[str, PlannerStep]
    predecessors: dict[str, set[str]]
    successors: dict[str, set[str]]
    in_degree: dict[str, int]

def extract_dependencies(dependency_list):
    if not dependency_list:
        return []
    return [d.ref for d in dependency_list]

def validate_graph(g : PlanDag):
    
    return 

def create_plan_dag(plan: PlannerResponse) -> PlanDag:
    steps = {step.evidence_id : step for step in plan.steps}
    predecessors = {idx : set() for idx in steps}
    successors = defaultdict(set)

    for step in plan.steps:
        evidence_id = step.evidence_id
        dependencies = extract_dependencies(step.dependencies if step.dependencies else [])
        for dependency in dependencies:
            if dependency == evidence_id:
                raise ValueError(
                    f"{evidence_id} cannot depend on itself"
                )
            successors[dependency].add(evidence_id)
            predecessors[evidence_id].add(dependency)

    in_degree = {
        evidence_id: len(dependencies)
        for evidence_id, dependencies in predecessors.items()
    }
    # validate_graph(successors, in_degree)

    return PlanDag(
        steps=steps,
        predecessors=predecessors,
        successors=successors ,
        in_degree=in_degree,
    )

In [ ]:
q1_zero_shot_response = PlannerResponse.model_validate_json(q1_zero_shot_chat_result.choices[0].message.content)
dag = create_plan_dag(q1_zero_shot_response)

# Plan Execution and Evidence Collection

Given a plan DAG. Use workers where possible to parallelize and collect the required evidences for the steps.
First look at the plan DAG, use the in_degree to get initial ready node set for immediate execution. As nodes are executed and evidence gathered, updates nodes to ready status when dependencies are met and execute parallely. Collect all evidences and send to solver.

In [ ]:
import asyncio
from collections.abc import Awaitable, Callable
from enum import Enum

In [ ]:
class NodeStatus(str, Enum):
    PENDING = "pending"
    RUNNING = "running"
    SUCCEEDED = "succeeded"
    FAILED = "failed"
    BLOCKED = "blocked"


@dataclass
class ExecutionResult:
    evidence: dict[str, Any] = field(default_factory=dict)
    statuses: dict[str, NodeStatus] = field(default_factory=dict)
    errors: dict[str, BaseException] = field(default_factory=dict)


ToolExecutor = Callable[[str, dict[str, Any]], Awaitable[Any]]

In [ ]:
# async def execute_plan(
#         plan: PlannerResponse,
#         exec_tool: ToolExecutor,
#         *,
#         max_concurrency: int=8
#     )-> ExecutionResult:
#     # build dag and initialize nodes for execution
#     dag = create_plan_dag(plan)
#     result = ExecutionResult(
#         statuses={
#             node:NodeStatus.PENDING for node in dag.steps
#         }
#     )
#     remaining = dict(dag.in_degree)
#     exec_queue : asyncio.Queue[str | None] = asyncio.Queue()
#     for node, degree in remaining.items():
#         # add items to execution queue if they have no in_degree (similar to a root node)
#         if degree == 0:
#             exec_queue.put_nowait(node)

#     completed = []
#     lock = asyncio.Lock()

#     async def update_children(node: str) -> None:
#         stack = list(dag.successors[node])
#         while stack:
#             current = stack.pop()
#             if result.statuses[current] != NodeStatus.PENDING:
#                 continue

#             result.statuses[current] = NodeStatus.BLOCKED
#             completed.append(current)
#             stack.extend(dag.successors[current])    

#     async def exec_shutdown() -> None:
#         return

    
#     async def worker() -> None:
#         while True:
#             node = await exec_queue.get()

#             try:
#                 if node is None:
#                     exec_queue.task_done()
#                     return
                
#                 async with lock:
#                     if result.statuses[node] != NodeStatus.PENDING:
#                         continue
#                     # if node status is pending set to running now
#                     result.statuses[node] = NodeStatus.RUNNING
#                 step = dag.steps[node]

#                 try:
#                     if step.tool_name == "construct_cohort_filter":
#                         output = exec_tool(step.tool_name, step.tool_arguments)
#                     else:
#                         output = await exec_tool(step.tool_name, step.tool_arguments)
#                 except Exception as error:
#                     async with lock:
#                         result.statuses[node]  = NodeStatus.FAILED
#                         result.errors[node] = error
#                         completed.append(node)

#                     await update_children(node)
#                     await exec_shutdown()

#                     continue

#                 async with lock:
#                     result.evidence[node] = output
#                     result.statuses[node] = NodeStatus.SUCCEEDED

#                     for child in dag.successors[node]:
#                         if result.statuses[child] != NodeStatus.PENDING:
#                             continue

#                         remaining[child] -= 1
#                         if remaining[child] == 0:
#                             exec_queue.put_nowait(child)

#                     await exec_shutdown()

#             finally:
#                 exec_queue.task_done()

#     # start workers
#     tasks = [asyncio.create_task(worker()) for _ in range(min(max_concurrency, len(dag.steps)))]    
#     await exec_queue.join()
#     for task in tasks:
#         task.cancel()
#     await asyncio.gather(*tasks, return_exceptions=True)

#     return result

## Tool Registry / Dispatcher

In [ ]:

from collections.abc import Callable, Mapping
from typing import Any

from pydantic import ConfigDict, ValidationError, validate_call

In [ ]:
import sys
import requests


import pandas as pd

from io import StringIO
from schema import GDCCohortSchema
from pydantic import ValidationError

GDC_API = "https://api.gdc.cancer.gov"

In [ ]:
def truncate(x, max_len=100):
    x = str(x)
    if len(x) > max_len:
        x = x[:max_len]
    return x

def gdc_query_all(
    endpoint: str,
    filters: dict,
    cohort_filter: dict | str | None = None,
    fields: list[str] | None = None,
    page_size: int = 1000,
) -> list[dict[str, Any]]:
    url = f"{GDC_API}/{endpoint}"
    all_hits = []
    offset = 0

    print(
        f"querying endpoint: {endpoint} with filters: {truncate(filters)} and cohort_filter: {json.loads(cohort_filter) if isinstance(cohort_filter, str) else cohort_filter}",
        file=sys.stderr,
    )

    all_filters = filters
    if cohort_filter:  # not None or empty string
        if isinstance(cohort_filter, str):
            cohort_filter = json.loads(cohort_filter)
        if filters:
            all_filters = {
                "op": "and",
                "content": [
                    cohort_filter,
                    filters,
                ],
            }
        else:
            all_filters = cohort_filter

    while True:
        payload = {
            "filters": all_filters,
            "from": offset,
            "size": page_size,
        }

        print(json.dumps(payload))
        if fields:
            # even if using post, fields needs to be a comma-separated string
            payload["fields"] = ",".join(fields)

        response = requests.post(url, json=payload)
        response.raise_for_status()
        resp_json = response.json()

        if resp_json["warnings"]:
            print(resp_json["warnings"], file=sys.stderr)

        data = resp_json["data"]

        hits = data["hits"]
        all_hits.extend(hits)

        pagination = data["pagination"]
        total = pagination["total"]

        # print(f"Retrieved {len(all_hits)} / {total}", file=sys.stderr)

        offset += page_size
        if offset >= total:
            break

    return all_hits

In [ ]:
CaseSet = Annotated[
    list[str],
    Field(
        description=(
            "A set of cases matching a specific cohort or query"
        ),
    ),
]
CaseCount = Annotated[
    int,
    Field(
        description="Number of cases for a retrieved or computed case set.",
    ),
]

Gene = Annotated[
    str,
    Field(
        description="Gene to query, for example 'BRAF'.",
    ),
]
AAChange = Annotated[
    str,
    Field(
        description="Amino acid change to query (in HGVS protein notation), for example 'V600E'.",
    ),
]
CNVChange = Annotated[
    Literal["gain", "amplification", "heterozygous deletion", "homozygous deletion"],
    Field(
        description="Copy number variation change to query. Valid options are 'gain', 'amplification', 'heterozygous deletion', or 'homozygous deletion'.",
    ),
]
MSIStatus = Annotated[
    Literal["msi", "mss"],
    Field(
        description="Microsatellite instability status. Valid options are 'msi' for microsatellite instable or 'mss' for microsatellite stable.",
    ),
]
Normalization = Annotated[
    Literal["uqfpkm", "median_centered_log2_uqfpkm"],
    Field(description="Gene expression normalization method."),
]
GenomicDataType = Annotated[
    Literal["ssm", "cnv", "msi", "gene_expression"],
    Field(description="Genomic data modality."),
]


## Available Tools

In [ ]:
from typing import Annotated

from pydantic import (
    BaseModel,
    BeforeValidator,
    ConfigDict,
    Field,
)


def normalize_string_list(value: object) -> list[str] | None:
    if value is None:
        return None

    if isinstance(value, str):
        value = value.strip()
        return [value] if value else []

    if isinstance(value, list):
        normalized: list[str] = []

        for item in value:
            if not isinstance(item, str):
                raise TypeError(
                    "Expected a string or a list of strings"
                )

            item = item.strip()

            if item:
                normalized.append(item)

        return normalized

    raise TypeError(
        "Expected a string, a list of strings, or None"
    )


StringList = Annotated[
    list[str] | None,
    BeforeValidator(normalize_string_list),
]

class ConstructCohortFilterArgs(BaseModel):
    model_config = ConfigDict(extra="forbid")
    project: StringList | None = None
    primary_site: StringList | None = None
    disease_type: StringList| None = None
    project_id: StringList | None = None
    sex: StringList | None = None
    smoking_status: StringList | None = None

    min_age: int | None = Field(default=None, ge=0)
    max_age: int | None = Field(default=None, ge=0)
    

class ResolverResponse(BaseModel):
    model_config = ConfigDict(extra="forbid")

    value: list[str]


def make_construct_cohort_filter_tool(
    *,
    client: Any,
    model: str,
    resolver_prompt: str,
    field_registry: Mapping[str, str],
    value_registry: Mapping[str, Any],
):
    async def _construct(
        args: ConstructCohortFilterArgs,
    ) -> GDCCohortSchema:
        supplied_fields = {
            field: value
            for field, value in args.model_dump().items()
            if value is not None
        }

        resolved_fields: dict[str, Any] = {}

        for field, value in supplied_fields.items():
            if field in {"min_age", "max_age"}:
                resolved_fields[field] = int(value * 365)
                continue

            if field not in field_registry:
                raise ValueError(
                    f"Unknown cohort field {field!r}"
                )

            if field not in value_registry:
                raise ValueError(
                    f"No allowed-value registry for {field!r}"
                )

            response = await asyncio.to_thread(
                client.chat.completions.create,
                model=model,
                messages=[
                    {
                        "role": "user",
                        "content": resolver_prompt.format(
                            source=value,
                            values=value_registry[field],
                        ),
                    }
                ],
                temperature=1.0,
                max_tokens=2048,
                extra_body={
                    "structured_outputs": {
                        "json": ResolverResponse.model_json_schema(),
                    }
                },
            )

            content = response.choices[0].message.content

            if content is None:
                raise ValueError(
                    f"Resolver returned no value for {field!r}"
                )

            resolver_result = (
                ResolverResponse.model_validate_json(content)
            )
            resolved_fields[field_registry[field]] = (
                resolver_result.value
            )

        sub_filters: list[dict[str, Any]] = []

        for field, value in resolved_fields.items():
            if field in {"min_age", "max_age"}:
                sub_filters.append(
                    {
                        "op": ">=" if field == "min_age" else "<=",
                        "content": {
                            "field": field_registry[field],
                            "value": value,
                        },
                    }
                )
            else:
                sub_filters.append(
                    {
                        "op": "in",
                        "content": {
                            "field": field,
                            "value": value,
                        },
                    }
                )

        return GDCCohortSchema(
            op="and",
            content=sub_filters
        )

    async def construct_cohort_filter(
        project: StringList | None = None,
        primary_site: StringList | None = None,
        disease_type: StringList | None = None,
        project_id: StringList | None = None,
        sex: StringList | None = None,
        smoking_status: StringList | None = None,
        min_age: int | None = None,
        max_age: int | None = None,
    ) -> str:
        args = ConstructCohortFilterArgs(
            project=project,
            primary_site=primary_site,
            disease_type=disease_type,
            project_id=project_id,
            sex=sex,
            min_age=min_age,
            max_age=max_age,
            smoking_status=smoking_status
        )

        return await _construct(args)

    return construct_cohort_filter

# def construct_cohort_filter():
#     try:
#         args =  json.loads(json.dumps(tool_arguments))
#     except ValidationError as e:
#         retry_reason.append(e)

#     cohort = []
#     for k, v in args.items():
#         print(f"key:{k}, value:{v}, value_type:{type(v)}")
#         if v:
#             cohort.append((k, v))

#     # resolve cohort
#     resolved_cohorts = {}

#     for c in cohort:
#         field, value = c
#         if field in ["min_age", "max_age"]:
#             resolved_cohorts[field] = int(value * 365) if value is not None else None
#             continue
        
#         _resolver_response = client.chat.completions.create(
#             model=model,
#             messages=[
#                 {
#                     "role": "user",
#                     "content": resolver_prompt.format(source=value, values=value_registry.get(field, "")),
#                 }
#             ],
#             temperature=1.0,
#             max_tokens=2048,
#             extra_body={
#                 "structured_outputs": {
#                     "json": ResolverResponse.model_json_schema(),
#                 }
#             },
#         )
#         resolved_cohorts[field_registry.get(field, "")] = json.loads(_resolver_response.choices[0].message.content)

#     # create filter
#     sub_filters = []
#     for c in resolved_cohorts.items():
#         field, value = c
#         if field in ["min_age", "max_age"]:
#             age_map = {
#                 "min_age": ">=",
#                 "max_age": "<="
#             }
#             sub_filters.append(
#                 {
#                     "op": age_map[field],
#                     "content": {
#                         "field": field_registry.get(field, ""),
#                         "value": value # direct int can be added
#                     }
#                 }
#             )
#         else:
#             sub_filters.append(
#                 {
#                     "op": "in",
#                     "content": {
#                         "field": field,
#                         "value": value["value"]
#                     }
#                 }
#             )
#     _cohort_filter = json.dumps({"op": "and", "content":sub_filters})
#     # # query against gdc
#     # hits = gdc_query_all(
#     #         endpoint="cases",
#     #         filters={},
#     #         cohort_filter=_cohort_filter,
#     #         fields=["project.project_id", "submitter_id"],
#     #     )
#     # case_ids = set([hit["id"] for hit in hits])

#     return _cohort_filter

In [ ]:
def get_cases_by_cohort_filter(cohort_filter: GDCCohortSchema) -> CaseSet:
    """
    A tool to query the GDC API for cases matching a given cohort filter.
    """
    print(
        f"get_cases_by_filter(cohort_filter={repr(cohort_filter)})",
        file=sys.stderr,
    )
    hits = gdc_query_all(
        endpoint="cases",
        filters={},
        cohort_filter=cohort_filter,
        fields=["project.project_id", "submitter_id"],
    )
    case_ids = set([hit["id"] for hit in hits])
    return case_ids

def compute_case_intersection(
        case_set_A: CaseSet,
        case_set_B: CaseSet,
) -> CaseSet:
    """
    A tool to compute the intersection between two case sets.
    """
    cases_A = set(case_set_A)
    cases_B = set(case_set_B)

    A_and_B = cases_A & cases_B

    return list(A_and_B)

def compute_case_union(
        case_set_A: CaseSet,
        case_set_B: CaseSet,
) -> CaseSet:
    """
    A tool to compute the union between two case sets.
    """

    cases_A = set(case_set_A)
    cases_B = set(case_set_B)

    A_and_B = cases_A | cases_B

    return list(A_and_B)

def get_case_set_size(
        case_set: CaseSet,
) -> CaseCount:
    """
    A tool to retireve the size of a precomputed case set.
    """

    return len(case_set)

In [ ]:
def get_simple_somatic_mutation_occurrences(
    cohort_filter: GDCCohortSchema,
    gene: Gene,
    aa_change: AAChange | None = None,
) -> CaseSet:
    """
    A tool to query the GDC API for cases with simple somatic mutations (SSMs).
    This tool searches within a specific cohort for SSMs within a given gene, optionally resulting in an amino acid (AA) change.
    The resulting case set is cached server side and can be referenced using the unique identifier returned by this tool.
    """
    print(
        f"get_simple_somatic_mutation_occurrences(gene={repr(gene)}, aa_change={repr(aa_change)}, cohort_filter={repr(cohort_filter)})",
        file=sys.stderr,
    )

    if gene is None:
        raise ValidationError(
            f"Cannot retrieve simple somatic mutation cases without a specific gene"
        )

    # first get SSMs for the given query parameters, either by gene & AA change or just gene
    if aa_change is not None:
        _filters = {
            "op": "in",
            "content": {
                "field": "gene_aa_change",
                "value": [f"{gene} {aa_change}"],  # e.g. 'BRAF V600E'
            },
        }
    else:
        _filters = {
            "op": "in",
            "content": {
                "field": "consequence.transcript.gene.symbol",
                "value": [gene],
            },
        }
    cohort_filter = cohort_filter.model_dump(mode="json")
    hits = gdc_query_all(
        endpoint="ssms",
        filters=_filters,
        cohort_filter=cohort_filter,
        fields=["gene_aa_change", "ssm_id"],
    )

    # then get cases matching those SSMs
    ssm_ids = [hit["ssm_id"] for hit in hits]
    ssm_id_filter = {
        "op": "in",
        "content": {
            "field": "ssm.ssm_id",
            "value": ssm_ids,
        },
    }
    hits = gdc_query_all(
        endpoint="ssm_occurrences",
        filters=ssm_id_filter,
        cohort_filter=cohort_filter,
        fields=["ssm.ssm_id", "case.submitter_id", "case.case_id"],
    )
    case_ids = set([hit["case"]["case_id"] for hit in hits])

    return list(case_ids)

In [ ]:
def get_copy_number_variant_occurrences(
    cohort_filter: GDCCohortSchema,
    gene: Gene | None = None,
    cnv_change: CNVChange | None = None,
) -> CaseSet:
    """
    A tool to query the GDC API for cases with copy number variations (CNVs).
    This tool searches within a specific cohort for CNVs, optionally within a given gene, optionally for a specific change type.
    If only the gene is specified, this tool retrieves all cases in the cohort with any change type for the given gene.
    If only the change type is specified, this tool retrieves all cases in the cohort with the given change type in any gene.
    It is an error to use this tool without at least one of gene or change type.
    """
    print(
        f"get_copy_number_variant_occurrences(gene={repr(gene)}, cnv_change={repr(cnv_change)}, cohort_filter={repr(cohort_filter)})",
        file=sys.stderr,
    )

    if gene is None and cnv_change is None:
        raise ValidationError(
            f"Cannot retrieve copy number variation cases without either a specific gene, a change type, or both"
        )

    # apparently `heterozygous deletion` change type is `loss` so remap it
    # keep the enum for the LLM to help it disambiguate the options
    # see: https://github.com/uc-cdis/gdc-qag/blob/07af83ecae960bd6f62347c473268ff73e725325/check_6k_mutations_withTable2.py#L541-L559
    if cnv_change == "heterozygous deletion":
        cnv_change = "loss"  # type: ignore

    ops = []
    if gene is not None:
        ops.append(
            {
                "op": "in",
                "content": {
                    "field": "cnv.consequence.gene.symbol",
                    "value": [gene],
                },
            },
        )
    if cnv_change is not None:
        ops.append(
            {
                "op": "in",
                "content": {
                    "field": "cnv.cnv_change_5_category",
                    "value": [cnv_change],
                },
            }
        )

    cohort_filter = cohort_filter.model_dump(mode="json")
    hits = gdc_query_all(
        endpoint="cnv_occurrences",
        filters={
            "op": "and",
            "content": ops,
        },
        cohort_filter=cohort_filter,
        fields=[
            "cnv.cnv_change",
            "cnv.cnv_change_5_category",
            "cnv.consequence.gene.symbol",
            "case.submitter_id",
            "case.case_id",
        ],
    )
    case_ids = set([hit["case"]["case_id"] for hit in hits])

    return list(case_ids)

In [ ]:
def get_microsatellite_instability_occurrences(
        cohort_filter: GDCCohortSchema,
        msi_status: MSIStatus,
) -> CaseSet:
    """
    A tool to query the GDC API for cases with microsatellite instability (MSI).
    This tool searches within a specific cohort for either MSI or MSS status.
    """
    print(
        f"get_microsatellite_instability_occurrences(msi_status={repr(msi_status)}, cohort_filter={repr(cohort_filter)})",
        file=sys.stderr,
    )
    cohort_filter = cohort_filter.model_dump(mode="json")
    # MSI is stored at the file level, so we need to query the files and aggregate back up to the cases
    hits = gdc_query_all(
        endpoint="files",
        filters={
            "op": "and",
            "content": [
                {
                    "op": "in",
                    "content": {
                        "field": "msi_status",
                        "value": [msi_status],
                    },
                },
                {"op": "in", "content": {"field": "data_format", "value": ["BAM"]}},
            ],
        },
        cohort_filter=cohort_filter,
        fields=[
            "msi_status",
            "msi_score",
            "cases.submitter_id",
            "cases.case_id",
        ],
    )

    case_ids = set()
    for hit in hits:
        for case in hit.get("cases", []):
            case_ids.add(case["case_id"])

    return list(case_ids)

In [ ]:
def get_cases_with_available_genomic_data(
        genomic_data_type: GenomicDataType,
        cohort_filter: GDCCohortSchema,
    ) -> CaseSet:
    cohort_filter = cohort_filter.model_dump(mode="json")
    if genomic_data_type == "ssm":
        hits = gdc_query_all(
            endpoint="case_ssms",
            filters={
                "op": "in",
                "content": {"field": "available_variation_data", "value": ["ssm"]},
            },
            cohort_filter=cohort_filter,
            fields=["project.project_id", "submitter_id"],
        )
        case_ids = set([hit["id"] for hit in hits])
        return case_ids
    elif genomic_data_type == "cnv" or genomic_data_type == "gene_expression":
        data_type_value = {
            "cnv": "Allele-specific Copy Number Segment",
            "gene_expression": "Gene Expression Quantification",
        }[genomic_data_type]
        hits = gdc_query_all(
            endpoint="cases",
            filters={
                "op": "and",
                "content": [
                    {
                        "op": "in",
                        "content": {
                            "field": "files.data_type",
                            "value": [data_type_value],
                        },
                    },
                    {
                        "op": "in",
                        "content": {
                            "field": "files.access",
                            "value": ["open"],
                        },
                    },
                ],
            },
            cohort_filter=cohort_filter,
            fields=["project.project_id", "submitter_id"],
        )
        case_ids = set([hit["id"] for hit in hits])
        return case_ids
    elif genomic_data_type == "msi":
        _msi_ids = get_microsatellite_instability_occurrences(cohort_filter, "msi")
        _mss_ids = get_microsatellite_instability_occurrences(cohort_filter, "mss")
        _ms_any_ids = compute_case_union(_msi_ids, _mss_ids)
        return _ms_any_ids
    else:
        raise ToolError(
            f"Unknown how to find available cases for given genomic data type: {genomic_data_type}"
        )


In [ ]:
def _get_ensembl_gene_ids(genes: list[Gene]) -> dict[str, Gene]:
    # convert standard gene names to ensembl IDs as a mapping of ensembl ID -> input name
    ensembl_gene_ids = dict()
    for gene in genes:
        filters = {
            "op": "and",
            "content": [{"op": "=", "content": {"field": "symbol", "value": gene}}],
        }
        try:
            response = requests.get(
                url=f"{GDC_API}/genes",
                params={
                    "filters": json.dumps(filters),
                    "fields": ",".join(["gene_id"]),
                    "size": 100,
                },
            )
            data = response.json()

            ensembl_id = data["data"]["hits"][0]["gene_id"]
            ensembl_gene_ids[ensembl_id] = gene
        except Exception as e:
            print("unable to execute GDC API request {}".format(str(e)))
    return ensembl_gene_ids

def get_gene_expression_statistics(
    case_set: CaseSet,
    genes: list[Gene],
    normalization: Normalization,
) -> dict[str, dict[str, float]]:
    """
    A tool to query the GDC API for gene expression statistics for a list of genes within a set of cases.
    Gene expression values are normalized either using FPKM-UQ or Median-centered log2(FPKM-UQ+1).
    """
    
    if len(genes) == 0:
        raise ValidationError("Must provide at least one gene")


    # compute and cache raw expression values
    gene_mapping = _get_ensembl_gene_ids(genes)
    response = requests.post(
        url=f"{GDC_API}/gene_expression/values",
        headers={
            "accept": "text/tab-separated-values",
            "Content-Type": "application/json",
        },
        json={
            "case_ids": case_set,
            "gene_ids": list(gene_mapping),
            "tsv_units": normalization,
            "format": "tsv",
        },
    )
    expr_str = response.text
    df = pd.read_csv(StringIO(expr_str), sep="\t")

    # map back to user-given names/order as best we can
    df["gene_id"] = [gene_mapping[g] for g in df["gene_id"]]
    df = df.set_index("gene_id")
    df = df.loc[list(gene_mapping.values())]

    summary = df.T.describe().to_dict()
    return summary  # type: ignore

def get_most_variably_expressed_genes(
    case_set: CaseSet,
    top_k: int = 1,
) -> list[Gene]:
    """
    A tool to query GDC API for the top-k most variably expressed genes within a set of cases.'
    """
    # do query
    response = requests.post(
        url=f"{GDC_API}/gene_expression/gene_selection",
        headers={
            "accept": "application/json",
            "Content-Type": "application/json",
        },
        json={
            "case_ids": case_set,
            "selection_size": top_k,
            "gene_type": "protein_coding",
        },
    )
    if not response.ok:
        raise ValidationError(f"GDC API returned status {response.status_code}")
    data = response.json()
    genes = []
    if "gene_selection" not in data:
        raise ValidationError("GDC API did not return genes")
    for item in data["gene_selection"]:
        if "symbol" not in item:
            print(f"Gene did not have symbol: {item}")
            continue
        genes.append(item["symbol"])

    return genes

## defines

In [ ]:
PROJECT_VALUES = """
"APOLLO-LUAD",
"APOLLO-OV",
"BEATAML1.0-COHORT",
"BEATAML1.0-CRENOLANIB",
"CDDP_EAGLE-1",
"CGCI-BLGSP",
"CGCI-HTMCP-CC",
"CGCI-HTMCP-DLBCL",
"CGCI-HTMCP-LC",
"CMI-ASC",
"CMI-MBC",
"CMI-MPC",
"CPTAC-2",
"CPTAC-3",
"CTSP-DLBCL1",
"EXCEPTIONAL_RESPONDERS-ER",
"FM-AD",
"HCMI-CMDC",
"MATCH-B",
"MATCH-C1",
"MATCH-H",
"MATCH-I",
"MATCH-N",
"MATCH-P",
"MATCH-Q",
"MATCH-R",
"MATCH-S1",
"MATCH-S2",
"MATCH-U",
"MATCH-W",
"MATCH-Y",
"MATCH-Z1A",
"MATCH-Z1B",
"MATCH-Z1D",
"MATCH-Z1I",
"MMRF-COMMPASS",
"MP2PRT-ALL",
"MP2PRT-WT",
"NCICCR-DLBCL",
"OHSU-CNL",
"ORGANOID-PANCREATIC",
"REBC-THYR",
"TARGET-ALL-P1",
"TARGET-ALL-P2",
"TARGET-ALL-P3",
"TARGET-AML",
"TARGET-CCSK",
"TARGET-NBL",
"TARGET-OS",
"TARGET-RT",
"TARGET-WT",
"TCGA-ACC",
"TCGA-BLCA",
"TCGA-BRCA",
"TCGA-CESC",
"TCGA-CHOL",
"TCGA-COAD",
"TCGA-DLBC",
"TCGA-ESCA",
"TCGA-GBM",
"TCGA-HNSC",
"TCGA-KICH",
"TCGA-KIRC",
"TCGA-KIRP",
"TCGA-LAML",
"TCGA-LGG",
"TCGA-LIHC",
"TCGA-LUAD",
"TCGA-LUSC",
"TCGA-MESO",
"TCGA-OV",
"TCGA-PAAD",
"TCGA-PCPG",
"TCGA-PRAD",
"TCGA-READ",
"TCGA-SARC",
"TCGA-SKCM",
"TCGA-STAD",
"TCGA-TGCT",
"TCGA-THCA",
"TCGA-THYM",
"TCGA-UCEC",
"TCGA-UCS",
"TCGA-UVM",
"TRIO-CRU",
"VAREPOP-APOLLO",
"WCDT-MCRPC",
"""
PRIMARY_SITE_VALUES = """
"accessory sinuses",
"adrenal gland",
"anus and anal canal",
"base of tongue",
"bladder",
"bones, joints and articular cartilage of limbs",
"bones, joints and articular cartilage of other and unspecified sites",
"brain",
"breast",
"bronchus and lung",
"cervix uteri",
"colon",
"connective, subcutaneous and other soft tissues",
"corpus uteri",
"esophagus",
"eye and adnexa",
"floor of mouth",
"gallbladder",
"gum",
"heart, mediastinum, and pleura",
"hematopoietic and reticuloendothelial systems",
"hypopharynx",
"kidney",
"larynx",
"lip",
"liver and intrahepatic bile ducts",
"lymph nodes",
"meninges",
"nasal cavity and middle ear",
"nasopharynx",
"not reported",
"oropharynx",
"other and ill-defined digestive organs",
"other and ill-defined sites",
"other and ill-defined sites in lip, oral cavity and pharynx",
"other and ill-defined sites within respiratory system and intrathoracic organs",
"other and unspecified female genital organs",
"other and unspecified major salivary glands",
"other and unspecified male genital organs",
"other and unspecified parts of biliary tract",
"other and unspecified parts of mouth",
"other and unspecified parts of tongue",
"other and unspecified urinary organs",
"other endocrine glands and related structures",
"ovary",
"palate",
"pancreas",
"parotid gland",
"penis",
"peripheral nerves and autonomic nervous system",
"prostate gland",
"rectosigmoid junction",
"rectum",
"renal pelvis",
"retroperitoneum and peritoneum",
"skin",
"small intestine",
"spinal cord, cranial nerves, and other parts of central nervous system",
"stomach",
"testis",
"thymus",
"thyroid gland",
"tonsil",
"trachea",
"unknown",
"ureter",
"uterus, nos",
"vagina",
"vulva",
"""
DISEASE_TYPE_VALUES = """
"acinar cell neoplasms",
"acute lymphoblastic leukemia",
"adenomas and adenocarcinomas",
"adnexal and skin appendage neoplasms",
"basal cell neoplasms",
"blood vessel tumors",
"chronic myeloproliferative disorders",
"complex epithelial neoplasms",
"complex mixed and stromal neoplasms",
"cystic, mucinous and serous neoplasms",
"ductal and lobular neoplasms",
"epithelial neoplasms, nos",
"fibroepithelial neoplasms",
"fibromatous neoplasms",
"germ cell neoplasms",
"gliomas",
"leukemias, nos",
"lipomatous neoplasms",
"lymphoid leukemias",
"mature b-cell lymphomas",
"mature t- and nk-cell lymphomas",
"meningiomas",
"mesothelial neoplasms",
"miscellaneous bone tumors",
"miscellaneous tumors",
"mucoepidermoid neoplasms",
"myelodysplastic syndromes",
"myeloid leukemias",
"myomatous neoplasms",
"neoplasms, nos",
"nerve sheath tumors",
"neuroepitheliomatous neoplasms",
"nevi and melanomas",
"not applicable",
"not reported",
"osseous and chondromatous neoplasms",
"paragangliomas and glomus tumors",
"plasma cell tumors",
"soft tissue tumors and sarcomas, nos",
"specialized gonadal neoplasms",
"squamous cell neoplasms",
"synovial-like neoplasms",
"thymic epithelial neoplasms",
"transitional cell papillomas and carcinomas",
"unknown",
"""
SEX_VALUES = """
"male", 
"female", 
"unknown"
"""
SMOKING_STATUS_VALUES="""
"current reformed smoker for > 15 yrs",
"current reformed smoker for < or = 15 yrs",
"current reformed smoker, duration not specified",
"current smoker",
"lifelong non-smoker",
"not reported",
"smoking history not documented",
"unknown",
"""
print(len(tokenizer.encode(PROJECT_VALUES)))
print(len(tokenizer.encode(PRIMARY_SITE_VALUES)))
print(len(tokenizer.encode(DISEASE_TYPE_VALUES)))
print(len(tokenizer.encode(SEX_VALUES)))
print(len(tokenizer.encode(SMOKING_STATUS_VALUES)))

In [ ]:
field_registry = {
    "project": "cases.project.project_id",
    "primary_site": "cases.primary_site",
    "disease_type": "cases.disease_type",
    "sex": "cases.demographic.sex_at_birth",
    "min_age": "cases.diagnoses.age_at_diagnosis",
    "max_age": "cases.diagnoses.age_at_diagnosis",
    "smoking_status": "cases.exposures.tobacco_smoking_status",
}
value_registry = {
    "project": PROJECT_VALUES,
    "primary_site": PRIMARY_SITE_VALUES,
    "disease_type": DISEASE_TYPE_VALUES,
    "sex": SEX_VALUES,
    "smoking_status": SMOKING_STATUS_VALUES,
}

tool_registry = {
    "get_cases_by_cohort_filter": get_cases_by_cohort_filter,
    "compute_case_intersection": compute_case_intersection,
    "compute_case_union": compute_case_union,
    "get_cases_with_available_genomic_data": get_cases_with_available_genomic_data,
    "get_simple_somatic_mutation_occurrences": get_simple_somatic_mutation_occurrences,
    "get_copy_number_variant_occurrences": get_copy_number_variant_occurrences,
    "get_microsatellite_instability_occurrences": get_microsatellite_instability_occurrences,
    "get_gene_expression_statistics": get_gene_expression_statistics,
    "get_most_variably_expressed_genes": get_most_variably_expressed_genes,
}

## working

In [ ]:
import inspect
for func_name, func in tool_registry.items():
    # print(f"name:{func_name},sig:{inspect.signature(func)}")
    # print(inspect.signature(func).parameters)
    d = inspect.signature(func).parameters
    print(d)
    tool_sig_types = [v.annotation for k,v in d.items()]
    print(tool_sig_types)
    break

In [ ]:
resolver_prompt = """
You are an AI assitant designed to pick matches for the source value from a list of possible values.
Pick the closest value possible. 
Source Value: {source}
Possible Values: {values}
RULES:
1. You must pick a value from the provided list of possible values
2. You must only pick one value.
3. Return the value exactly as it is presented to you.
4. Return only the value, do not provide any explanation for why you chose it.
## BEGIN ##
"""
# class ResolverResponse(BaseModel):
#     value: str = Field(
#         min_length=1, 
#         max_length = 256,
#         description="Value that best matches the source value"
#     )

In [ ]:
import re

from pydantic import ConfigDict, ValidationError, validate_call


EVIDENCE_REF_PATTERN = re.compile(r"^#E[1-9]\d*$")


class ToolExecutionError(Exception):
    pass

class UnknownToolError(ToolExecutionError):
    pass

class UnknownEvidenceReferenceError(ToolExecutionError):
    pass


class InvalidToolArgumentsError(Exception):
    def __init__(
        self,
        tool_name: str,
        validation_error: ValidationError,
    ) -> None:
        self.tool_name = tool_name
        self.validation_error = validation_error

        super().__init__(
            f"Invalid arguments for tool {tool_name!r}: "
            f"{validation_error}"
        )

def resolve_evidence_refs(
    value: Any,
    evidence_store: Mapping[str, Any],
) -> Any:
    if isinstance(value, str) and EVIDENCE_REF_PATTERN.fullmatch(value):
        try:
            return evidence_store[value]
        except KeyError as exc:
            raise UnknownEvidenceReferenceError(
                f"Evidence reference {value!r} has not been produced"
            ) from exc

    if isinstance(value, Mapping):
        return {
            key: resolve_evidence_refs(item, evidence_store)
            for key, item in value.items()
        }

    if isinstance(value, list):
        return [
            resolve_evidence_refs(item, evidence_store)
            for item in value
        ]

    if isinstance(value, tuple):
        return tuple(
            resolve_evidence_refs(item, evidence_store)
            for item in value
        )

    return value


class ToolExecutor:
    def __init__(self) -> None:
        self._tools: dict[str, Callable[..., Any]] = {}

    def register(
        self,
        tool: Callable[..., Any],
        *,
        name: str | None = None,
    ) -> None:
        tool_name = name or tool.__name__

        if tool_name in self._tools:
            raise ValueError(
                f"Tool {tool_name!r} is already registered"
            )

        validated_tool = validate_call(
            config=ConfigDict(
                extra="forbid",
                arbitrary_types_allowed=True,
            )
        )(tool)

        self._tools[tool_name] = validated_tool

    async def __call__(
        self,
        tool_name: str,
        tool_arguments: Mapping[str, Any],
        *,
        evidence_store: Mapping[str, Any],
    ) -> Any:
        try:
            tool = self._tools[tool_name]
        except KeyError as exc:
            raise UnknownToolError(
                f"Unknown tool {tool_name!r}. "
                f"Available tools: {sorted(self._tools)}"
            ) from exc

        if not isinstance(tool_arguments, Mapping):
            raise InvalidToolArgumentsError(
                tool_name,
                TypeError(
                    "Tool arguments must be a mapping, "
                    f"received {type(tool_arguments).__name__}"
                ),
            )

        resolved_arguments = resolve_evidence_refs(
            tool_arguments,
            evidence_store,
        )

        try:
            if inspect.iscoroutinefunction(tool):
                return await tool(**resolved_arguments)

            # Run synchronous tools outside the event loop.
            return await asyncio.to_thread(
                tool,
                **resolved_arguments,
            )

        except ValidationError as exc:
            raise InvalidToolArgumentsError(
                tool_name,
                exc,
            ) from exc

In [ ]:
executor = ToolExecutor()
construct_cohort_filter = make_construct_cohort_filter_tool(
    client=client,
    model=model,
    resolver_prompt=resolver_prompt,
    field_registry=field_registry,
    value_registry=value_registry,
)

executor.register(construct_cohort_filter, name="construct_cohort_filter")
executor.register(get_cases_by_cohort_filter)
executor.register(compute_case_intersection)
executor.register(compute_case_union)
executor.register(get_case_set_size)
executor.register(get_cases_with_available_genomic_data)
executor.register(get_simple_somatic_mutation_occurrences)
executor.register(get_copy_number_variant_occurrences)
executor.register(get_microsatellite_instability_occurrences)
executor.register(get_gene_expression_statistics)
executor.register(get_most_variably_expressed_genes)


In [ ]:
async def execute_plan(
    plan: PlannerResponse,
    exec_tool: ToolExecutor,
    *,
    max_concurrency: int = 8,
) -> ExecutionResult:
    dag = create_plan_dag(plan)

    result = ExecutionResult(
        statuses={
            node: NodeStatus.PENDING
            for node in dag.steps
        }
    )

    remaining = dict(dag.in_degree)
    exec_queue: asyncio.Queue[str | None] = asyncio.Queue()

    for node, degree in remaining.items():
        if degree == 0:
            exec_queue.put_nowait(node)

    lock = asyncio.Lock()

    async def block_descendants(failed_node: str) -> None:
        stack = list(dag.successors[failed_node])

        async with lock:
            while stack:
                current = stack.pop()

                if result.statuses[current] != NodeStatus.PENDING:
                    continue

                result.statuses[current] = NodeStatus.BLOCKED
                stack.extend(dag.successors[current])

    async def worker() -> None:
        while True:
            node = await exec_queue.get()

            try:
                if node is None:
                    return

                async with lock:
                    if result.statuses[node] != NodeStatus.PENDING:
                        continue

                    result.statuses[node] = NodeStatus.RUNNING

                step = dag.steps[node]

                try:
                    output = await exec_tool(
                        step.tool_name,
                        step.tool_arguments,
                        evidence_store=result.evidence,
                    )
                    print(output)
                except Exception as error:
                    async with lock:
                        result.statuses[node] = NodeStatus.FAILED
                        result.errors[node] = error

                    await block_descendants(node)
                    continue

                async with lock:
                    result.evidence[node] = output
                    result.statuses[node] = NodeStatus.SUCCEEDED

                    for child in dag.successors[node]:
                        if result.statuses[child] != NodeStatus.PENDING:
                            continue

                        remaining[child] -= 1

                        if remaining[child] == 0:
                            exec_queue.put_nowait(child)

            finally:
                exec_queue.task_done()

    worker_count = min(
        max_concurrency,
        len(dag.steps),
    )

    tasks = [
        asyncio.create_task(worker())
        for _ in range(worker_count)
    ]

    await exec_queue.join()

    for task in tasks:
        task.cancel()

    await asyncio.gather(
        *tasks,
        return_exceptions=True,
    )

    return result


In [ ]:
# def run_tool(tool_name: str, tool_arguments: Dict):
#     retry_reason = []
#     if tool_name == "construct_cohort_filter":
#         try:
#             args =  json.loads(json.dumps(tool_arguments))
#         except ValidationError as e:
#             retry_reason.append(e)

#         cohort = []
#         for k, v in args.items():
#             print(f"key:{k}, value:{v}, value_type:{type(v)}")
#             if v:
#                 cohort.append((k, v))

#         # resolve cohort
#         resolved_cohorts = {}

#         for c in cohort:
#             field, value = c
#             if field in ["min_age", "max_age"]:
#                 resolved_cohorts[field] = int(value * 365) if value is not None else None
#                 continue
            
#             _resolver_response = client.chat.completions.create(
#                 model=model,
#                 messages=[
#                     {
#                         "role": "user",
#                         "content": resolver_prompt.format(source=value, values=value_registry.get(field, "")),
#                     }
#                 ],
#                 temperature=1.0,
#                 max_tokens=2048,
#                 extra_body={
#                     "structured_outputs": {
#                         "json": ResolverResponse.model_json_schema(),
#                     }
#                 },
#             )
#             resolved_cohorts[field_registry.get(field, "")] = json.loads(_resolver_response.choices[0].message.content)

#         # create filter
#         sub_filters = []
#         for c in resolved_cohorts.items():
#             field, value = c
#             if field in ["min_age", "max_age"]:
#                 age_map = {
#                     "min_age": ">=",
#                     "max_age": "<="
#                 }
#                 sub_filters.append(
#                     {
#                         "op": age_map[field],
#                         "content": {
#                             "field": field_registry.get(field, ""),
#                             "value": value # direct int can be added
#                         }
#                     }
#                 )
#             else:
#                 sub_filters.append(
#                     {
#                         "op": "in",
#                         "content": {
#                             "field": field,
#                             "value": value["value"]
#                         }
#                     }
#                 )
#         _cohort_filter = json.dumps({"op": "and", "content":sub_filters})
#         # # query against gdc
#         # hits = gdc_query_all(
#         #         endpoint="cases",
#         #         filters={},
#         #         cohort_filter=_cohort_filter,
#         #         fields=["project.project_id", "submitter_id"],
#         #     )
#         # case_ids = set([hit["id"] for hit in hits])
    
#         return _cohort_filter
#     else:
#         try:
#             use_tool = tool_registry[tool_name]
#         except Exception as e:
#             retry_reason.append(e)

#         # validate the tool arguments
#         # first match tool arguments to tool type
#         # then gather base models for validation
#         # resolve any references to cohorts and retrieve
#         valid, error =  validate_kwargs(use_tool, tool_arguments)
#         if not valid:
#             retry_reason.append(error)
#         else:
#             # tool args shows valid argument names
#             # check arg types now
#             tool_sig = inspect.signature(use_tool).parameters
#             tool_sig_types = [v.annotation for k,v in tool_sig.items()]
            
#             return result 



# Re-solver

In [ ]:
class Resolver:
    def __init__(
        self,
        plan: PlanDag,
    ):
        super().__init__()
        self.plan = plan

    def extract_cohort(self):
        self.cohort = []
        for step in self.plan.steps:
            step_plan = self.plan.steps[step]
            if step_plan.tool_name == "construct_cohort_filter":
                step_arguments = step_plan.tool_arguments
                for k, v in step_arguments.items():
                    if v is not None:
                        self.cohort.append((k, v))

        return self.cohort

    def run(
        self,
    ):
        return

In [ ]:
q1_zero_shot_response = PlannerResponse.model_validate_json(q1_zero_shot_chat_result.choices[0].message.content)
q1_dag = create_plan_dag(q1_zero_shot_response)

In [ ]:
q1_resolver_response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "user",
            "content": resolver_prompt.format(source=curr_cohort[0][1], values=PROJECT_VALUES),
        }
    ],
    temperature=1.0,
    max_tokens=2048,
    extra_body={
        "structured_outputs": {
            "json": ResolverResponse.model_json_schema(),
        }
    },
)
json.loads(q1_resolver_response.choices[0].message.content)

## q2

In [ ]:
q2_zero_shot_response = PlannerResponse.model_validate_json(q2_zero_shot_chat_result.choices[0].message.content)
q2_dag = create_plan_dag(q2_zero_shot_response)

In [ ]:
q2_dag.steps["#E1"]

In [ ]:
print(q2_dag.steps["#E2"].tool_name)
print(q2_dag.steps["#E2"].tool_arguments)

In [ ]:
tool_name = q2_dag.steps["#E2"].tool_name
tool_arguments = q2_dag.steps["#E2"].tool_arguments

In [ ]:
# validate_kwargs(tool_registry[tool_name], tool_arguments)

In [ ]:
use_tool  = tool_registry[tool_name]
# use_tool(**tool_arguments)

In [ ]:
q2_result = await execute_plan(q2_zero_shot_response, executor)

In [ ]:
q2_result.statuses

In [ ]:
q2_result.errors

In [ ]:
q2_dag.steps["#E3"]

In [ ]:
resolver = Resolver(q2_dag)
curr_cohort = resolver.extract_cohort()
print(curr_cohort)

In [ ]:
results = []
for cohort in curr_cohort:
    field, value = cohort
    if field in ["min_age", "max_age"]:
        continue
    q2_resolver_response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "user",
                "content": resolver_prompt.format(source=value, values=value_registry.get(field, "")),
            }
        ],
        temperature=1.0,
        max_tokens=2048,
        extra_body={
            "structured_outputs": {
                "json": ResolverResponse.model_json_schema(),
            }
        },
    )
    results.append(json.loads(q2_resolver_response.choices[0].message.content))

In [ ]:
results

## q3

In [ ]:
q3_zero_shot_response = PlannerResponse.model_validate_json(q3_zero_shot_chat_result.choices[0].message.content)
q3_dag = create_plan_dag(q3_zero_shot_response)

In [ ]:
q3_result = await execute_plan(q3_zero_shot_response, executor)

In [ ]:
q3_result.statuses

In [ ]:
q3_result.errors

In [ ]:
resolver = Resolver(q3_dag)
curr_cohort = resolver.extract_cohort()
print(curr_cohort)

In [ ]:
results = []
for cohort in curr_cohort:
    field, value = cohort
    if field in ["min_age", "max_age"]:
        continue
    q3_resolver_response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "user",
                "content": resolver_prompt.format(source=value, values=value_registry.get(field, "")),
            }
        ],
        temperature=1.0,
        max_tokens=2048,
        extra_body={
            "structured_outputs": {
                "json": ResolverResponse.model_json_schema(),
            }
        },
    )
    results.append(json.loads(q3_resolver_response.choices[0].message.content))

In [ ]:
results

## q4

In [ ]:
q4_zero_shot_response = PlannerResponse.model_validate_json(q4_zero_shot_chat_result.choices[0].message.content)
q4_dag = create_plan_dag(q4_zero_shot_response)

In [ ]:
q4_result = await execute_plan(q4_zero_shot_response, executor)

In [ ]:
q4_result.statuses

In [ ]:
q4_result

In [ ]:
q4_result.errors

In [ ]:
resolver = Resolver(q4_dag)
curr_cohort = resolver.extract_cohort()
print(curr_cohort)

In [ ]:
results = []
for cohort in curr_cohort:
    field, value = cohort
    if field in ["min_age", "max_age"]:
        continue
    q4_resolver_response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "user",
                "content": resolver_prompt.format(source=value, values=value_registry.get(field, "")),
            }
        ],
        temperature=1.0,
        max_tokens=2048,
        extra_body={
            "structured_outputs": {
                "json": ResolverResponse.model_json_schema(),
            }
        },
    )
    results.append(json.loads(q4_resolver_response.choices[0].message.content))

In [ ]:
results

## q5

In [ ]:
q5_zero_shot_response = PlannerResponse.model_validate_json(q5_zero_shot_chat_result.choices[0].message.content)
q5_dag = create_plan_dag(q5_zero_shot_response)

In [ ]:
q5_result = await execute_plan(q5_zero_shot_response, executor)

In [ ]:
q5_result

In [ ]:
q5_result.statuses

In [ ]:
q5_result.errors

In [ ]:
resolver = Resolver(q5_dag)
curr_cohort = resolver.extract_cohort()
print(curr_cohort)

In [ ]:
results = []
for cohort in curr_cohort:
    field, value = cohort
    if field in ["min_age", "max_age"]:
        continue
    q5_resolver_response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "user",
                "content": resolver_prompt.format(source=value, values=value_registry.get(field, "")),
            }
        ],
        temperature=1.0,
        max_tokens=2048,
        extra_body={
            "structured_outputs": {
                "json": ResolverResponse.model_json_schema(),
            }
        },
    )
    results.append(json.loads(q5_resolver_response.choices[0].message.content))

In [ ]:
results

## q6

In [ ]:
q6_zero_shot_response = PlannerResponse.model_validate_json(q6_zero_shot_chat_result.choices[0].message.content)
q6_dag = create_plan_dag(q6_zero_shot_response)

In [ ]:
q6_result = await execute_plan(q6_zero_shot_response, executor)

In [ ]:
q6_result.statuses

In [ ]:
resolver = Resolver(q6_dag)
curr_cohort = resolver.extract_cohort()
print(curr_cohort)

In [ ]:
results = []
for cohort in curr_cohort:
    field, value = cohort
    if field in ["min_age", "max_age"]:
        continue
    q6_resolver_response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "user",
                "content": resolver_prompt.format(source=value, values=value_registry.get(field, "")),
            }
        ],
        temperature=1.0,
        max_tokens=2048,
        extra_body={
            "structured_outputs": {
                "json": ResolverResponse.model_json_schema(),
            }
        },
    )
    results.append(json.loads(q6_resolver_response.choices[0].message.content))

In [ ]:
results

In [ ]:
q6_dag.steps

In [ ]:
from pydantic import ValidationError
def run_tool(tool_name: str, tool_arguments: Dict):
    retry_reason = []
    if tool_name == "construct_cohort_filter":
        try:
            args =  json.loads(json.dumps(tool_arguments))
        except ValidationError as e:
            retry_reason.append(e)

        cohort = []
        for k, v in args.items():
            print(f"key:{k}, value:{v}, value_type:{type(v)}")
            if v:
                cohort.append((k, v))

        # resolve cohort
        resolved_cohorts = {}

        for c in cohort:
            field, value = c
            if field in ["min_age", "max_age"]:
                resolved_cohorts[field] = int(value * 365) if value is not None else None
                continue
            
            _resolver_response = client.chat.completions.create(
                model=model,
                messages=[
                    {
                        "role": "user",
                        "content": resolver_prompt.format(source=value, values=value_registry.get(field, "")),
                    }
                ],
                temperature=1.0,
                max_tokens=2048,
                extra_body={
                    "structured_outputs": {
                        "json": ResolverResponse.model_json_schema(),
                    }
                },
            )
            resolved_cohorts[field_registry.get(field, "")] = json.loads(_resolver_response.choices[0].message.content)

        print(f"resolved cohort: {resolved_cohorts}")
        # create filter
        sub_filters = []
        for c in resolved_cohorts.items():
            field, value = c
            if field in ["min_age", "max_age"]:
                age_map = {
                    "min_age": ">=",
                    "max_age": "<="
                }
                sub_filters.append(
                    {
                        "op": age_map[field],
                        "content": {
                            "field": field_registry.get(field, ""),
                            "value": value # direct int can be added
                        }
                    }
                )
            else:
                sub_filters.append(
                    {
                        "op": "in",
                        "content": {
                            "field": field,
                            "value": value["value"]
                        }
                    }
                )
        _cohort_filter = json.dumps({"op": "and", "content":sub_filters})
        # query against gdc
        hits = gdc_query_all(
                endpoint="cases",
                filters={},
                cohort_filter=_cohort_filter,
                fields=["project.project_id", "submitter_id"],
            )
        case_ids = set([hit["id"] for hit in hits])
    
        return case_ids
    else:
        try:
            use_tool = tool_registry[tool_name]
        except Exception as e:
            retry_reason.append(e)


# Solver

In [ ]:
q6_result.statuses

In [ ]:
q6_result

In [ ]:
type(q6_result.evidence)

In [ ]:
ZeroShotSolverPrompt = """
You are an AI assitant who solves problems given a concrete step by step plan and evidences for each step.
Your task is to briefly state the problem, the steps takens to arrive at a solution and then using the evidence
make a short final conclusion for your task. The main task is a Q&A about accurate variant statistics for the GDC.

##Original Task##
{task}

##Plan & Evidence##
{plan_evidence}

##Begin##
"""

In [ ]:
q6_dag.steps

In [ ]:
for step in q6_dag.steps.items():
    

In [ ]:
def create_plan_evidence_history(
        plan_dag: PlanDag, 
        execution_result: ExecutionResult
    ):
    result = []
    for i, step in enumerate(plan_dag.steps.items()):
        evidence_id, step_ = step
        evi_dict = execution_result.evidence
        evidence_ = evi_dict[evidence_id]
        try:
            evidence_ = evidence_.model_dump(mode="json")
        except Exception as e:
            evidence_ = evidence_
        result.append(
            {
                f"#S{i+1}": f"{step_.plan}",
                evidence_id: evidence_
            }
        )
    return result

In [ ]:
q6_plan_evidence = create_plan_evidence_history(q6_dag, q6_result)

In [ ]:
q6_plan_evidence

In [ ]:
q6_task = "Compare the expression of TP53, MUC16, and BRAF in APOLLO-LUAD."
q6_solver_result = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "user",
            "content": ZeroShotSolverPrompt.format(plan_evidence=q6_plan_evidence, task=q6_task),
        }
    ],
    temperature=1.0,
    max_tokens=2048,
    seed=42,
)

In [ ]:
print(q6_solver_result.choices[0].message.content)